In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import argparse

# 绝对路径
FOLD_PATH = r"D:\Project_Github\audio_click_mil\processed_data\folds\fold_0.json"
FEATURE_BASE = r"D:\Project_Github\audio_click_mil\processed_data\features"
LABEL_CSV = r"D:\Project_Github\audio_click_mil\processed_data\balanced_bag_labels.csv"
GT_CSVS = [
    r"D:\Project_Github\audio_click_mil\data\ClickTrains.csv",
    r"D:\Project_Github\audio_click_mil\data\BurstPulseTrains.csv",
    r"D:\Project_Github\audio_click_mil\data\BuzzTrains.csv"
]
RESULT_DIR = r"D:\Project_Github\audio_click_mil\processed_data\results"
os.makedirs(RESULT_DIR, exist_ok=True)

class BagDataset(Dataset):
    def __init__(self, bag_list, feature_method):
        self.bags = []
        self.labels = []
        self.file_nums = []
        self.bag_idxs = []
        feat_dir = os.path.join(FEATURE_BASE, feature_method)
        for bag_info in bag_list:
            fname = f"file_{bag_info['file_num']}_bag_{bag_info['bag_idx']:03d}_label_{bag_info['bag_label']}_feat.npy"
            fpath = os.path.join(feat_dir, fname)
            feat = np.load(fpath)  # (60, 32)
            self.bags.append(torch.tensor(feat, dtype=torch.float32))
            self.labels.append(bag_info['bag_label'])
            self.file_nums.append(bag_info['file_num'])
            self.bag_idxs.append(bag_info['bag_idx'])
    
    def __len__(self):
        return len(self.bags)
    
    def __getitem__(self, idx):
        return self.bags[idx], torch.tensor(self.labels[idx], dtype=torch.float32)

class SimpleMIL(nn.Module):
    def __init__(self, feat_dim=32):
        super().__init__()
        self.proj = nn.Linear(feat_dim, 128)
        self.attn = nn.Linear(128, 1)
        self.classifier = nn.Linear(128, 1)
    
    def forward(self, bag):  # bag: (60, 32)
        h = torch.relu(self.proj(bag))  # (60, 128)
        logits = self.attn(h).squeeze(-1)  # (60,)
        attn_weights = torch.softmax(logits, dim=0)
        bag_rep = torch.sum(attn_weights.unsqueeze(-1) * h, dim=0)  # (128,)
        logit = self.classifier(bag_rep).squeeze()
        return logit, attn_weights

def load_bag_list(fold_data, split, label_df):
    file_list = fold_data[split + "_files"]
    if isinstance(file_list[0], str):
        file_list = [int(f) for f in file_list]
    bag_list = []
    for _, row in label_df.iterrows():
        if int(row['file_num']) in file_list:
            bag_list.append({
                'file_num': int(row['file_num']),
                'bag_idx': int(row['bag_idx']),
                'bag_label': int(row['bag_label'])
            })
    return bag_list

def compute_instance_label(file_num, bag_start_sec, instance_idx, gt_dfs):
    instance_start = bag_start_sec + instance_idx
    instance_end = instance_start + 1.0
    for df in gt_dfs:
        gt = df[df['Ori_file_num(No.)'] == file_num]
        for _, row in gt.iterrows():
            if max(instance_start, row['Train_start(s)']) < min(instance_end, row['Train_end(s)']):
                return 1
    return 0

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--feature_method', type=str, default='mfcc', choices=['teager', 'wavelet', 'mfcc', 'logmel'])
    args = parser.parse_args()
    
    # 加载fold和标签
    with open(FOLD_PATH, 'r') as f:
        fold = json.load(f)
    label_df = pd.read_csv(LABEL_CSV)
    gt_dfs = [pd.read_csv(p) for p in GT_CSVS]
    
    train_bags = load_bag_list(fold, 'train', label_df)
    test_bags = load_bag_list(fold, 'test', label_df)
    
    train_ds = BagDataset(train_bags, args.feature_method)
    test_ds = BagDataset(test_bags, args.feature_method)
    
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)  # bag级
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    
    model = SimpleMIL()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    
    print("开始训练（10 epochs）...")
    for epoch in tqdm(range(10), desc="Epoch进度"):
        model.train()
        for bag, label in train_loader:
            optimizer.zero_grad()
            logit, _ = model(bag.squeeze(0))
            loss = criterion(logit, label)
            loss.backward()
            optimizer.step()
    
    # 测试与结果收集
    model.eval()
    sheet0_data = []
    sheet1_data = []
    
    with torch.no_grad():
        for i, (bag, label) in enumerate(tqdm(test_loader, desc="测试集推理进度")):
            logit, attn = model(bag.squeeze(0))
            prob = torch.sigmoid(logit).item()
            pred_label = 1 if prob > 0.5 else 0
            true_label = label.item()
            
            bag_info = test_bags[i]
            sheet0_data.append({
                'bag_id': f"file_{bag_info['file_num']}_bag_{bag_info['bag_idx']}",
                'true_label': true_label,
                'pred_label': pred_label,
                'prob': round(prob, 4)
            })
            
            # sheet1: 每个instance
            bag_start_sec = label_df[(label_df['file_num'] == bag_info['file_num']) & 
                                     (label_df['bag_idx'] == bag_info['bag_idx'])]['bag_start_sec'].values[0]
            for inst_idx in range(60):
                orig_label = compute_instance_label(bag_info['file_num'], bag_start_sec, inst_idx, gt_dfs)
                attn_val = attn[inst_idx].item()
                pred_inst = 1 if attn_val > 0.5 else 0
                sheet1_data.append({
                    'bag_id': f"file_{bag_info['file_num']}_bag_{bag_info['bag_idx']}",
                    'instance_idx': inst_idx,
                    'original_instance_label': orig_label,
                    'attention_weight': round(attn_val, 4),
                    'predicted_instance_label': pred_inst
                })
    
    # 输出结果（Excel带sheet）
    result_path = os.path.join(RESULT_DIR, f"mil_results_{args.feature_method}.xlsx")
    with pd.ExcelWriter(result_path, engine='openpyxl') as writer:
        pd.DataFrame(sheet0_data).to_excel(writer, sheet_name='sheet0', index=False)
        pd.DataFrame(sheet1_data).to_excel(writer, sheet_name='sheet1', index=False)
    
    print(f"训练与评估完成，结果已保存至 {result_path}（sheet0: bag级标签；sheet1: instance级标签与注意力）")

usage: ipykernel_launcher.py [-h]
                             [--feature_method {teager,wavelet,mfcc,logmel}]
ipykernel_launcher.py: error: argument --feature_method: invalid choice: 'c:\\Users\\Chico\\AppData\\Roaming\\jupyter\\runtime\\kernel-v32c40cf3e320e679e127551e18e16a51ddea409c2.json' (choose from teager, wavelet, mfcc, logmel)


SystemExit: 2

d:\Python_env\Dolphin\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
